# ModernBERT Model Comparison & Analysis

This notebook evaluates and compares various ModernBERT models trained for hate speech detection. 
It focuses on:
1.  **Dynamic Model Discovery**: Automatically finding all trained models in the directory.
2.  **Slavic Hate Detection**: Evaluating performance specifically on Slavic hate speech.
3.  **Generalization**: Evaluating performance on other target groups.
4.  **Advanced Analysis**: Error analysis, confidence distributions, and correlation matrices.
5.  **Density Analysis**: Correlating model performance with training data density.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, confusion_matrix, precision_recall_fscore_support
from tqdm.auto import tqdm
from typing import List, Dict, Tuple

# Setup Plotting Style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Configuration
BASE_MODEL_DIR = "models_knn_tox_refactored"
DATA_DIR = "new_data"
SLAVIC_TEST_FILE = os.path.join(DATA_DIR, "russ_annot_masked.csv")
BASELINE_MODEL_ID = "answerdotai/ModernBERT-base"

## 1. Dynamic Model Discovery

In [ ]:
def find_models(base_dir: str) -> List[Dict]:
    """Finds all trained models in the directory structure."""
    models = []
    
    # Walk through the directory
    for root, dirs, files in os.walk(base_dir):
        # Check if it looks like a model directory (has config.json or model.safetensors)
        if "config.json" in files or "model.safetensors" in files:
            # Skip tokenizer directories if they are separate
            if "tokenizer" in root.lower():
                continue
                
            # Determine Model Type and Name based on path
            # Expected structure: .../knn/k5/final or .../ensemble/asian/final
            path_parts = root.replace("\\", "/").split("/")
            
            model_type = "Unknown"
            variant = "Unknown"
            
            if "knn" in path_parts:
                model_type = "KNN"
                # Try to find the k-value (e.g., k5, k100)
                for part in path_parts:
                    if part.startswith("k") and part[1:].isdigit():
                        variant = part
                        break
            elif "ensemble" in path_parts:
                model_type = "Ensemble-Term"
                # The term is usually the folder before 'final' or the model folder itself
                if path_parts[-1] == "final":
                    variant = path_parts[-2]
                else:
                    variant = path_parts[-1]
            
            # If we found a valid model, add it
            if model_type != "Unknown":
                models.append({
                    "name": f"{model_type}_{variant}",
                    "path": root,
                    "type": model_type,
                    "variant": variant
                })
    
    return models

discovered_models = find_models(BASE_MODEL_DIR)
print(f"Found {len(discovered_models)} models:")
for m in discovered_models:
    print(f" - {m['name']} ({m['path']})")

## 2. Data Loading

In [ ]:
def load_slavic_data(path):
    """Loads the Slavic test dataset."""
    df = pd.read_csv(path)
    # Ensure binary labels
    if df['label'].dtype == 'object':
        df['label'] = df['label'].apply(lambda x: 1 if x == 'hate' else 0)
    return df

def load_general_data(data_dir, samples_per_group=100):
    """Loads a stratified sample of other target groups for generalization testing."""
    all_dfs = []
    # List of known identity terms
    terms = ['asian', 'black', 'chinese', 'jewish', 'latino', 'lgbtq', 
             'mental_dis', 'mexican', 'middle_east', 'muslim', 
             'native_american', 'physical_dis', 'women']
    
    for term in terms:
        path = os.path.join(data_dir, f"{term}.csv")
        if os.path.exists(path):
            df = pd.read_csv(path)
            # Basic cleaning
            if 'label' in df.columns:
                if df['label'].dtype == 'object':
                    df['label'] = df['label'].apply(lambda x: 1 if x == 'hate' else 0)
                
                # Sample to keep evaluation fast but representative
                if len(df) > samples_per_group:
                    df = df.sample(n=samples_per_group, random_state=42)
                
                df['target_group'] = term
                all_dfs.append(df[['text', 'label', 'target_group', 'density']])
    
    if not all_dfs:
        return pd.DataFrame()
        
    return pd.concat(all_dfs, ignore_index=True)

# Load Datasets
slavic_df = load_slavic_data(SLAVIC_TEST_FILE)
general_df = load_general_data(DATA_DIR)

print(f"Slavic Test Set: {len(slavic_df)} samples")
print(f"General Test Set: {len(general_df)} samples (across {general_df['target_group'].nunique()} groups)")

## 3. Evaluation Logic

In [ ]:
def evaluate_model(model, tokenizer, df, batch_size=32):
    """Evaluates a single model on a dataframe."""
    texts = df['text'].tolist()
    labels = df['label'].tolist()
    
    preds = []
    probs = []
    
    model.eval()
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            batch_probs = torch.nn.functional.softmax(outputs.logits, dim=1)
            
        preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
        probs.extend(batch_probs[:, 1].cpu().numpy())
        
    # Metrics
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='binary')
    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = 0.5
        
    return {
        "Accuracy": acc,
        "F1": f1,
        "AUC": auc,
        "Predictions": preds,
        "Probabilities": probs
    }

def load_and_evaluate(model_path, model_name, slavic_df, general_df):
    """Loads a model and runs evaluation on both datasets."""
    print(f"Evaluating {model_name}...")
    try:
        # Handle tokenizer path (sometimes it's in a separate 'tokenizer' folder, sometimes same as model)
        tokenizer_path = model_path
        if os.path.exists(os.path.join(model_path, "..", "tokenizer")):
             tokenizer_path = os.path.join(model_path, "..", "tokenizer")
        elif os.path.exists(os.path.join(model_path, "tokenizer")):
             tokenizer_path = os.path.join(model_path, "tokenizer")
             
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
        
        # Eval on Slavic
        slavic_res = evaluate_model(model, tokenizer, slavic_df)
        
        # Eval on General
        general_res = evaluate_model(model, tokenizer, general_df)
        
        # Clean up
        del model
        del tokenizer
        torch.cuda.empty_cache()
        
        return {
            "Model": model_name,
            "Slavic_F1": slavic_res["F1"],
            "Slavic_AUC": slavic_res["AUC"],
            "General_F1": general_res["F1"],
            "General_AUC": general_res["AUC"],
            "Slavic_Probs": slavic_res["Probabilities"],
            "General_Probs": general_res["Probabilities"]
        }
    except Exception as e:
        print(f"Error evaluating {model_name}: {e}")
        return None

## 4. Run Evaluation Loop

In [ ]:
results = []

# 1. Evaluate Baseline (Untuned)
print("Evaluating Baseline...")
try:
    tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_ID)
    model = AutoModelForSequenceClassification.from_pretrained(BASELINE_MODEL_ID).to(device)
    
    slavic_res = evaluate_model(model, tokenizer, slavic_df)
    general_res = evaluate_model(model, tokenizer, general_df)
    
    results.append({
        "Model": "Baseline (Untuned)",
        "Slavic_F1": slavic_res["F1"],
        "Slavic_AUC": slavic_res["AUC"],
        "General_F1": general_res["F1"],
        "General_AUC": general_res["AUC"],
        "Slavic_Probs": slavic_res["Probabilities"],
        "General_Probs": general_res["Probabilities"]
    })
    
    del model
    del tokenizer
    torch.cuda.empty_cache()
except Exception as e:
    print(f"Error evaluating baseline: {e}")

# 2. Evaluate Discovered Models
for m in tqdm(discovered_models, desc="Evaluating Models"):
    res = load_and_evaluate(m['path'], m['name'], slavic_df, general_df)
    if res:
        results.append(res)

# 3. Evaluate Ensemble (if components exist)
ensemble_models = [m for m in discovered_models if m['type'] == 'Ensemble-Term']
if ensemble_models:
    print("Evaluating Ensemble...")
    # Logic to load all ensemble models and average predictions
    # This is a simplified version of the ensemble logic
    ensemble_probs_slavic = np.zeros(len(slavic_df))
    ensemble_probs_general = np.zeros(len(general_df))
    count = 0
    
    for m in tqdm(ensemble_models, desc="Ensemble Components"):
        res = load_and_evaluate(m['path'], m['name'], slavic_df, general_df)
        if res:
            ensemble_probs_slavic += np.array(res["Slavic_Probs"])
            ensemble_probs_general += np.array(res["General_Probs"])
            count += 1
            
    if count > 0:
        ensemble_probs_slavic /= count
        ensemble_probs_general /= count
        
        # Calculate Metrics
        slavic_preds = (ensemble_probs_slavic >= 0.5).astype(int)
        general_preds = (ensemble_probs_general >= 0.5).astype(int)
        
        results.append({
            "Model": "Ensemble (Average)",
            "Slavic_F1": f1_score(slavic_df['label'], slavic_preds),
            "Slavic_AUC": roc_auc_score(slavic_df['label'], ensemble_probs_slavic),
            "General_F1": f1_score(general_df['label'], general_preds),
            "General_AUC": roc_auc_score(general_df['label'], ensemble_probs_general),
            "Slavic_Probs": ensemble_probs_slavic.tolist(),
            "General_Probs": ensemble_probs_general.tolist()
        })

# Create Results DataFrame
results_df = pd.DataFrame(results)
display_cols = ["Model", "Slavic_F1", "Slavic_AUC", "General_F1", "General_AUC"]
print(results_df[display_cols].sort_values("Slavic_F1", ascending=False))

## 5. Analysis & Visualization

In [ ]:
# 1. Model Comparison Plots
def plot_comparison(df, metric="Slavic_F1", title="Model Comparison"):
    plt.figure(figsize=(14, 6))
    sns.barplot(data=df.sort_values(metric, ascending=False), x="Model", y=metric, palette="viridis")
    plt.xticks(rotation=45, ha="right")
    plt.title(title)
    plt.ylim(0, 1.0)
    plt.tight_layout()
    plt.show()

plot_comparison(results_df, "Slavic_F1", "Slavic Hate Detection Performance (F1)")
plot_comparison(results_df, "General_F1", "General Hate Detection Performance (F1)")

In [ ]:
# 2. Cross-Group Heatmap (Detailed Generalization)
# We need to re-evaluate the best model on each specific group in general_df

best_model_row = results_df.sort_values("Slavic_F1", ascending=False).iloc[0]
best_model_name = best_model_row["Model"]
print(f"Detailed Analysis for Best Model: {best_model_name}")

# Find path for best model
best_model_path = None
if best_model_name == "Baseline (Untuned)":
    best_model_path = BASELINE_MODEL_ID
elif best_model_name == "Ensemble (Average)":
    print("Skipping detailed group heatmap for Ensemble (requires re-loading all models)")
else:
    for m in discovered_models:
        if m['name'] == best_model_name:
            best_model_path = m['path']
            break

if best_model_path:
    # Load model once
    tokenizer_path = best_model_path
    if os.path.exists(os.path.join(best_model_path, "..", "tokenizer")):
         tokenizer_path = os.path.join(best_model_path, "..", "tokenizer")
    elif os.path.exists(os.path.join(best_model_path, "tokenizer")):
         tokenizer_path = os.path.join(best_model_path, "tokenizer")
         
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    model = AutoModelForSequenceClassification.from_pretrained(best_model_path).to(device)
    
    group_metrics = []
    for group in general_df['target_group'].unique():
        group_df = general_df[general_df['target_group'] == group]
        res = evaluate_model(model, tokenizer, group_df)
        group_metrics.append({"Group": group, "F1": res["F1"], "AUC": res["AUC"]})
        
    group_metrics_df = pd.DataFrame(group_metrics)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=group_metrics_df.sort_values("F1", ascending=False), x="Group", y="F1", palette="magma")
    plt.title(f"Performance by Target Group ({best_model_name})")
    plt.xticks(rotation=45)
    plt.ylim(0, 1.0)
    plt.show()
    
    del model
    del tokenizer
    torch.cuda.empty_cache()

In [ ]:
# 3. Prediction Correlation Matrix
# Check how similar the models are in their predictions on Slavic data

probs_dict = {}
for _, row in results_df.iterrows():
    if "Probs" in row and row["Slavic_Probs"] is not None:
        probs_dict[row["Model"]] = row["Slavic_Probs"]

if probs_dict:
    probs_df = pd.DataFrame(probs_dict)
    corr = probs_df.corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=0, vmax=1)
    plt.title("Model Prediction Correlation (Slavic Data)")
    plt.show()

In [ ]:
# 4. Density Analysis
# Visualize the density distribution of the training data (using the general_df which has density)

plt.figure(figsize=(14, 6))
sns.violinplot(data=general_df, x="target_group", y="density", palette="muted")
plt.title("Density Distribution by Target Group")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 5. Confidence Distribution
# For the best model, show confidence histogram split by True Label

if best_model_path:
    # We need the predictions again, or retrieve from results
    best_probs = results_df[results_df["Model"] == best_model_name]["Slavic_Probs"].values[0]
    
    plot_df = pd.DataFrame({
        "Probability": best_probs,
        "True Label": slavic_df["label"].map({0: "No Hate", 1: "Hate"})
    })
    
    plt.figure(figsize=(10, 6))
    sns.histplot(data=plot_df, x="Probability", hue="True Label", bins=20, kde=True, element="step")
    plt.title(f"Prediction Confidence Distribution ({best_model_name})")
    plt.xlabel("Predicted Probability of Hate")
    plt.show()